# News Data Split

This notebook randomly splits all records in `data/processed/final.jsonl` into:

- 80% training records
- 10% validation records
- 10% test records

The split is reproducible and each input record appears in exactly one output file.

## Setup

In [1]:
import os
import json
import pandas as pd

PROCESSED_DIR = "data/processed"
INPUT_PATH = os.path.join(PROCESSED_DIR, "final.jsonl")
TRAIN_PATH = os.path.join(PROCESSED_DIR, "train.jsonl")
VAL_PATH = os.path.join(PROCESSED_DIR, "validation.jsonl")
TEST_PATH = os.path.join(PROCESSED_DIR, "test.jsonl")

TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1
SEED = 42

assert TRAIN_RATIO + VAL_RATIO + TEST_RATIO == 1.0

## Load processed records

In [2]:
if not os.path.exists(INPUT_PATH):
    raise FileNotFoundError(
        f"Could not find '{INPUT_PATH}'. Create the final processed dataset first."
    )

records = []
with open(INPUT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

processed_df = pd.DataFrame(records)
if processed_df.empty:
    raise ValueError(f"No records were found in '{INPUT_PATH}'.")
if "article_id" not in processed_df.columns:
    raise ValueError("The processed dataset must contain an 'article_id' column.")
if processed_df["article_id"].duplicated().any():
    raise ValueError("The processed dataset contains duplicate article IDs.")

print(f"Loaded {len(processed_df)} records")

Loaded 500 records


## Create disjoint train, validation and test sets

In [3]:
shuffled_df = processed_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

total_records = len(shuffled_df)
train_size = int(total_records * TRAIN_RATIO)
val_size = int(total_records * VAL_RATIO)
train_end = train_size
val_end = train_end + val_size

train_df = shuffled_df.iloc[:train_end].copy().reset_index(drop=True)
val_df = shuffled_df.iloc[train_end:val_end].copy().reset_index(drop=True)
test_df = shuffled_df.iloc[val_end:].copy().reset_index(drop=True)

# Leakage checks.
assert set(train_df["article_id"]).isdisjoint(val_df["article_id"])
assert set(train_df["article_id"]).isdisjoint(test_df["article_id"])
assert set(val_df["article_id"]).isdisjoint(test_df["article_id"])
assert len(train_df) + len(val_df) + len(test_df) == total_records

print(f"Train:      {len(train_df)} records ({len(train_df) / total_records:.0%})")
print(f"Validation: {len(val_df)} records ({len(val_df) / total_records:.0%})")
print(f"Test:       {len(test_df)} records ({len(test_df) / total_records:.0%})")

Train:      400 records (80%)
Validation: 50 records (10%)
Test:       50 records (10%)


## Save splits

In [4]:
def save_jsonl(df: pd.DataFrame, path: str):
    with open(path, "w", encoding="utf-8") as f:
        for record in df.to_dict(orient="records"):
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    print(f"Saved {len(df)} rows to '{path}'")


save_jsonl(train_df, TRAIN_PATH)
save_jsonl(val_df, VAL_PATH)
save_jsonl(test_df, TEST_PATH)

Saved 400 rows to 'data/processed\train.jsonl'
Saved 50 rows to 'data/processed\validation.jsonl'
Saved 50 rows to 'data/processed\test.jsonl'
